# 🧠 Taller 5 SQL - Lógica en la BD PostgreSQL
## Gimnasia Rítmica

Este taller está diseñado para ejecutarse en Google Colaboratory usando una base de datos PostgreSQL. Aprenderás immplementar lógica en la BD, usandovistas, triggers y procedimientos almacenados

## Objetivos
- Crear una base de datos PostgreSQL en Colab.
- Crear vistas
- Crear tablas temporales
- Crear triggers al insertarse datos en la BD
- Implementar Procedimientos almacenados, que usen vistas, triggers y actualizand datos


In [ ]:
# Instalación y configuración de PostgreSQL
!apt update
!apt install postgresql postgresql-contrib -y
!service postgresql start
!sudo -u postgres psql -c "CREATE USER root WITH SUPERUSER"


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [81.0 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages

In [ ]:
# Cargar extensión SQL y conectar con PostgreSQL
%load_ext sql
%config SqlMagic.feedback=False
%config SqlMagic.autopandas=True
%sql postgresql+psycopg2://@/postgres


## 🏗️ Crear tablas

Creamos las tablas necesarias para modelar la base de datos de gimnasia rítmica. Incluye campos tipo `DATE` y relaciones entre entidades.


In [ ]:
%%sql

DROP TABLE IF EXISTS Evaluaciones, Presentaciones, Gimnastas, Conjuntos, Instrumentos, Campeonatos, Tipos, Equipos CASCADE;

CREATE TABLE Equipos (
    id_equipo SERIAL PRIMARY KEY,
    nombre_equipo TEXT
);

CREATE TABLE Conjuntos (
    id_conj SERIAL PRIMARY KEY,
    nombre_conj TEXT,
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Gimnastas (
    id_gim SERIAL PRIMARY KEY,
    nombre TEXT,
    fecha_nac DATE,
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_equipo INTEGER REFERENCES Equipos(id_equipo)
);

CREATE TABLE Instrumentos (
    id_inst SERIAL PRIMARY KEY,
    nombre_inst TEXT
);

CREATE TABLE Campeonatos (
    id_camp SERIAL PRIMARY KEY,
    nombre_camp TEXT,
    fecha_camp DATE
);

CREATE TABLE Tipos (
    id_tipo SERIAL PRIMARY KEY,
    nombre_tipo TEXT
);

CREATE TABLE Presentaciones (
    id_pres SERIAL PRIMARY KEY,
    id_gim INTEGER REFERENCES Gimnastas(id_gim),
    id_conj INTEGER REFERENCES Conjuntos(id_conj),
    id_inst INTEGER REFERENCES Instrumentos(id_inst),
    id_camp INTEGER REFERENCES Campeonatos(id_camp),
    id_tipo INTEGER REFERENCES Tipos(id_tipo)
);

CREATE TABLE Evaluaciones (
    id_eval SERIAL PRIMARY KEY,
    id_pres INTEGER REFERENCES Presentaciones(id_pres),
    pje_eje REAL,
    pje_dif REAL,
    pje_art REAL
);


 * postgresql+psycopg2://@/postgres


""


In [ ]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


## 📥 Insertar datos simulados

Insertamos datos ficticios para poblar la base de datos: 5 equipos, 10 conjuntos, 20 gimnastas, 15 campeonatos, 50 presentaciones individuales, 50 presentaciones de conjunto y 100 evaluaciones.


In [ ]:
%%sql

-- Insertar equipos
INSERT INTO Equipos (nombre_equipo) VALUES
('Estrellas'), ('Auroras'), ('Fénix'), ('Orion'), ('Galaxia');

-- Insertar conjuntos
INSERT INTO Conjuntos (nombre_conj, id_equipo) VALUES
('Estrellas A', 1), ('Estrellas B', 1), ('Auroras A', 2), ('Auroras B', 2),
('Fénix A', 3), ('Fénix B', 3), ('Orion A', 4), ('Orion B', 4),
('Galaxia A', 5), ('Galaxia B', 5);

-- Insertar gimnastas
INSERT INTO Gimnastas (nombre, fecha_nac, id_conj, id_equipo) VALUES
('Ana Torres', '2005-03-12', 1, 1), ('Lucía Pérez', '2006-07-25', 1, 1),
('María Gómez', '2004-11-05', 2, 1), ('Sofía Díaz', '2007-01-20', 2, 1),
('Valentina Ruiz', '2005-06-30', 3, 2), ('Camila Soto', '2006-09-15', 3, 2),
('Isabella León', '2004-12-01', 4, 2), ('Martina Ríos', '2007-04-10', 4, 2),
('Renata Silva', '2005-08-22', 5, 3), ('Emilia Vargas', '2006-10-05', 5, 3),
('Josefa Herrera', '2004-07-18', 6, 3), ('Antonia Castro', '2007-02-28', 6, 3),
('Florencia Peña', '2005-05-14', 7, 4), ('Amanda Fuentes', '2006-11-11', 7, 4),
('Julieta Navarro', '2004-09-09', 8, 4), ('Agustina Bravo', '2007-03-03', 8, 4),
('Daniela Pino', '2005-12-25', 9, 5), ('Bianca Morales', '2006-06-06', 9, 5),
('Carla Espinoza', '2004-10-10', 10, 5), ('Fernanda Reyes', '2007-07-07', 10, 5),
('Colomba Gómez', '2004-11-05', 2, 1), ('Montserrat Díaz', '2007-01-20', 2, 1);

-- Insertar instrumentos
INSERT INTO Instrumentos (nombre_inst) VALUES
('Cinta'), ('Aro'), ('Balón'), ('Manos libres'), ('Cuerda');

-- Insertar campeonatos
INSERT INTO Campeonatos (nombre_camp, fecha_camp) VALUES
('Campeonato Nacional', '2023-06-15'), ('Copa Primavera', '2023-09-10'),
('Copa Invierno', '2023-12-05'), ('Torneo Sur', '2023-08-20'),
('Copa Norte', '2023-07-01'), ('Copa Andes', '2023-10-10'),
('Copa Pacífico', '2023-11-11'), ('Copa Estrellas', '2024-05-05'),
('Copa Aurora', '2024-04-04'), ('Copa Fénix', '2025-03-03'),
('Copa Orion', '2025-02-02'), ('Copa Galaxia', '2025-01-01'),
('Copa Final', '2024-12-31'), ('Copa Apertura', '2025-01-15'),
('Copa Clausura', '2024-12-01');

-- Insertar tipos
INSERT INTO Tipos (nombre_tipo) VALUES ('Individual'), ('Conjunto');




 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
-- Insertar presentaciones
DO $$
DECLARE
    i INTEGER := 1;
    gim_id INTEGER;
    conj_id INTEGER;
    inst_id INTEGER;
    tipo_id INTEGER;
    camp_id INTEGER;
BEGIN
    WHILE i <= 100 LOOP
        -- Selección aleatoria de IDs válidos
        SELECT id_gim INTO gim_id FROM Gimnastas ORDER BY RANDOM() LIMIT 1;
        SELECT id_conj INTO conj_id FROM Conjuntos ORDER BY RANDOM() LIMIT 1;
        SELECT id_inst INTO inst_id FROM Instrumentos ORDER BY RANDOM() LIMIT 1;
        SELECT id_tipo INTO tipo_id FROM Tipos ORDER BY RANDOM() LIMIT 1;
        SELECT id_camp INTO camp_id FROM Campeonatos ORDER BY RANDOM() LIMIT 1;

        -- Inserción en la tabla Presentaciones
        INSERT INTO Presentaciones (id_gim, id_conj, id_inst, id_camp, id_tipo)
        VALUES (gim_id, conj_id, inst_id, camp_id, tipo_id);

        i := i + 1;
    END LOOP;
END $$;


 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
-- Insertar evaluaciones
INSERT INTO Evaluaciones (id_pres, pje_eje, pje_dif, pje_art)
SELECT p.id_pres,
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2),
       ROUND((RANDOM() * 5 + 5)::NUMERIC, 2)
FROM Presentaciones p
LIMIT 100;

 * postgresql+psycopg2://@/postgres


""


## 🔍 Ejercicios de Consultas SQL



1.- Cree una vista que entregue los lugares obtenidos en cada campeonato para la competencia individual, separado por tipo de presentación e instrumento, siendo el primer lugar el que haya obtenido mayor puntaje total.

In [ ]:
%%sql
create view lugares_individual as
select p.id_camp, c.nombre_camp, i.nombre_inst, t.nombre_tipo, g.nombre, (e.pje_eje+e.pje_dif+e.pje_art) as puntaje
from campeonatos c
join presentaciones p on c.id_camp = p.id_camp
join gimnastas g on p.id_gim = g.id_gim
join instrumentos i on p.id_inst = i.id_inst
join tipos t on p.id_tipo = t.id_tipo
join evaluaciones e on p.id_pres = e.id_pres
where t.nombre_tipo = 'Individual'
order by p.id_camp,i.nombre_inst, t.nombre_tipo, puntaje desc

 * postgresql+psycopg2://@/postgres


""


2.- Use la vista creada en el paso anterior y obtenga los lugares para el campenoato **Copa Pacífico**

In [ ]:
%%sql
select * from lugares_individual where id_camp = 7

 * postgresql+psycopg2://@/postgres


,id_camp,nombre_camp,nombre_inst,nombre_tipo,nombre,puntaje
0,7,Copa Pacífico,Cinta,Individual,Amanda Fuentes,20.51
1,7,Copa Pacífico,Cinta,Individual,Montserrat Díaz,19.34
2,7,Copa Pacífico,Manos libres,Individual,Isabella León,26.58


3.- Cree un procedimiento almacenado que reciba como parámetro el nombre un equipo y lo inserte en la tabla equipos

In [ ]:
%%sql
create or replace procedure crea_equipo(nombre_equipo_param text)
language plpgsql
as $$
begin
    insert into equipos (nombre_equipo) values (nombre_equipo_param);
end;
$$

 * postgresql+psycopg2://@/postgres


""


4.- Usando el procedimiento creado en el paso anterior, cree 2 equipos nuevos.

In [ ]:
%%sql
call crea_equipo('Equipo Nuevo 1');
call crea_equipo('Equipo Nuevo 2');

 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
select * from equipos;

 * postgresql+psycopg2://@/postgres


,id_equipo,nombre_equipo
0,1,Estrellas
1,2,Auroras
2,3,Fénix
3,4,Orion
4,5,Galaxia
5,6,Equipo Nuevo 1
6,7,Equipo Nuevo 2


5.- Cree un procedimiento almacenado que reciba como parámetros el nombre de un equipo, el nombre de una gimnasta y su fecha de nacimiento. El procedimiento busca el id del equipo y crea la gimnasta en la tabla de gimnastas. En caso de no existir el equipo, entrega un mensaje de error indicando que el equipo no fue encontrado

In [ ]:
%%sql
create or replace procedure crea_gimnasta(nombre_equipo_param text, nombre_gim_param text, fecha_nac_param date)
language plpgsql
as $$
declare
    id_equipo_param integer;
begin
    select id_equipo into id_equipo_param from equipos where nombre_equipo = nombre_equipo_param;
    if not found then
        raise exception 'Equipo no encontrado';
    else
        insert into gimnastas (nombre, fecha_nac, id_equipo) values (nombre_gim_param, fecha_nac_param, id_equipo_param);
    end if;
end;
$$

 * postgresql+psycopg2://@/postgres


""


6.- Use el procedimiento almacenado creado en el paso anterior y ejecútelo para los siguientes 2 casos y compruebe que funcione correctamente


1.   Use uno de los equipos creados en el paso 4
2.   Use un equipo que no exista



In [ ]:
%%sql
call crea_gimnasta('Equipo Nuevo 1', 'Gimnasta Nuevo 1', '2000-01-01');
call crea_gimnasta('Equipo Nuevo 3', 'Gimnasta Nuevo 2', '2000-01-01');

 * postgresql+psycopg2://@/postgres
(psycopg2.errors.RaiseException) Equipo no encontrado
CONTEXT:  PL/pgSQL function crea_gimnasta(text,text,date) line 7 at RAISE

[SQL: call crea_gimnasta('Equipo Nuevo 3', 'Gimnasta Nuevo 2', '2000-01-01');]
(Background on this error at: https://sqlalche.me/e/20/2j85)


In [ ]:
%%sql
select * from gimnastas;

 * postgresql+psycopg2://@/postgres


,id_gim,nombre,fecha_nac,id_conj,id_equipo
0,1,Ana Torres,2005-03-12,1.0,1
1,2,Lucía Pérez,2006-07-25,1.0,1
2,3,María Gómez,2004-11-05,2.0,1
3,4,Sofía Díaz,2007-01-20,2.0,1
4,5,Valentina Ruiz,2005-06-30,3.0,2
5,6,Camila Soto,2006-09-15,3.0,2
6,7,Isabella León,2004-12-01,4.0,2
7,8,Martina Ríos,2007-04-10,4.0,2
8,9,Renata Silva,2005-08-22,5.0,3
9,10,Emilia Vargas,2006-10-05,5.0,3


7.- Realice los siguientes pasos:
*   Cree una tabla que se llama tmp_lugares_invidual, que tenga los mismos campos y tipos de datos que devuelve la vista *lugares_individual*, más un campo tipo timestamp
*   Cree un trigger que cada_vez que se inserta un registro en la tabla de ***Evaluaciones***, inserte el resultado de la ejecución de la vista creada en el paso 1, en la tabla tmp_lugares_individual y en el campo timestamp la fecha y hora en que se ejecutó el trigger.

In [ ]:
%%sql
drop table if exists tmp_lugares_individual;
create table tmp_lugares_individual as
select *, CURRENT_TIMESTAMP AS fecha_registro
from lugares_individual
where 1=0;

 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
select * from tmp_lugares_individual

 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
create or replace function obtener_lugares()
returns trigger
language plpgsql
as $$
declare
    id_camp_param integer;
begin
    select id_camp into id_camp_param
    from presentaciones
    where id_pres = NEW.id_pres;

    insert into tmp_lugares_individual
    select *, CURRENT_TIMESTAMP
    from lugares_individual
    where id_camp=id_camp_param;
    return new;
end;
$$

 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
create or replace trigger actualizar_lugares
after insert on evaluaciones
for each row
execute function obtener_lugares();


 * postgresql+psycopg2://@/postgres


""


8.- Cree 2 registros en la tabla **Evaluaciones** para distintos campeonatos y verifique que funcionó correctamente

In [ ]:
%%sql
insert into evaluaciones (id_pres, pje_eje, pje_dif, pje_art) values (1, 4.5, 4.5, 4.5);
insert into evaluaciones (id_pres, pje_eje, pje_dif, pje_art) values (2, 4.5, 4.5, 4.5);

 * postgresql+psycopg2://@/postgres


""


In [ ]:
%%sql
select * from tmp_lugares_individual;


 * postgresql+psycopg2://@/postgres


,id_camp,nombre_camp,nombre_inst,nombre_tipo,nombre,puntaje,fecha_registro
0,12,Copa Galaxia,Aro,Individual,Martina Ríos,25.939999,2025-09-29 01:32:09.572196+00:00
1,12,Copa Galaxia,Aro,Individual,Florencia Peña,22.470000,2025-09-29 01:32:09.572196+00:00
2,12,Copa Galaxia,Cuerda,Individual,Isabella León,25.340000,2025-09-29 01:32:09.572196+00:00
3,8,Copa Estrellas,Cuerda,Individual,Antonia Castro,26.600000,2025-09-29 01:32:09.578636+00:00
4,8,Copa Estrellas,Cuerda,Individual,María Gómez,22.400002,2025-09-29 01:32:09.578636+00:00
5,8,Copa Estrellas,Cuerda,Individual,Camila Soto,20.570000,2025-09-29 01:32:09.578636+00:00
6,8,Copa Estrellas,Manos libres,Individual,Lucía Pérez,26.530000,2025-09-29 01:32:09.578636+00:00
7,8,Copa Estrellas,Manos libres,Individual,Lucía Pérez,13.500000,2025-09-29 01:32:09.578636+00:00
8,12,Copa Galaxia,Aro,Individual,Martina Ríos,25.939999,2025-09-29 01:33:13.660612+00:00
9,12,Copa Galaxia,Aro,Individual,Florencia Peña,22.470000,2025-09-29 01:33:13.660612+00:00
